# PGDiff Autoresearch — Experiment Analysis

Analyze autonomous hyperparameter + architecture tuning results from `results.tsv`.
Tracks guidance loss, per-term breakdown, and experiment outcomes.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import os, json

# Load TSV (8 columns: commit, avg_loss, loss_per_image, task, status, description, key_params, loss_breakdown)
df = pd.read_csv("results.tsv", sep="\t")
df["avg_loss"] = pd.to_numeric(df["avg_loss"], errors="coerce")
df["loss_per_image"] = pd.to_numeric(df["loss_per_image"], errors="coerce")
df["status"] = df["status"].str.strip().str.upper()

print(f"Total experiments: {len(df)}")
print(f"Columns: {list(df.columns)}")
print(f"Tasks: {df['task'].unique().tolist()}")
df.head(10)

In [ ]:
# Outcome distribution
counts = df["status"].value_counts()
print("Experiment outcomes:")
print(counts.to_string())

n_keep = counts.get("KEEP", 0)
n_discard = counts.get("DISCARD", 0)
n_crash = counts.get("CRASH", 0)
n_decided = n_keep + n_discard
if n_decided > 0:
    print(f"\nKeep rate: {n_keep}/{n_decided} = {n_keep / n_decided:.1%}")

print("\nOutcomes by task:")
print(pd.crosstab(df["task"], df["status"]))

In [ ]:
# Show all KEPT experiments with key params
kept = df[df["status"] == "KEEP"].copy()
print(f"KEPT experiments ({len(kept)} total):\n")
for i, (_, row) in enumerate(kept.iterrows()):
    print(f"  #{i:3d}  loss={row['avg_loss']:9.1f}  per_img={row['loss_per_image']:7.1f}  "
          f"task={row['task']:20s}  [{row['key_params']:30s}]  {row['description']}")

## Guidance Loss Over Time

Running minimum shows the frontier — best result so far. Lower is better.

In [ ]:
fig, ax = plt.subplots(figsize=(18, 9))

valid = df[df["status"] != "CRASH"].copy().reset_index(drop=True)
baseline_loss = valid.loc[0, "avg_loss"]

# Discarded as faint dots
disc = valid[valid["status"] == "DISCARD"]
ax.scatter(disc.index, disc["avg_loss"], c="#cccccc", s=12, alpha=0.5, zorder=2, label="Discarded")

# Kept as prominent dots — color by task
kept_v = valid[valid["status"] == "KEEP"]
tasks_uniq = kept_v["task"].unique()
colors = ["#2ecc71", "#3498db", "#e74c3c", "#f39c12", "#9b59b6"]
for task, c in zip(tasks_uniq, colors[:len(tasks_uniq)]):
    t_data = kept_v[kept_v["task"] == task]
    ax.scatter(t_data.index, t_data["avg_loss"], c=c, s=60, zorder=4,
               label=f"Kept: {task}", edgecolors="black", linewidths=0.5)

# Running minimum
kept_mask = valid["status"] == "KEEP"
kept_idx = valid.index[kept_mask]
kept_loss_vals = valid.loc[kept_mask, "avg_loss"]
running_min = kept_loss_vals.cummin()
ax.step(kept_idx, running_min, where="post", color="#27ae60",
        linewidth=2.5, alpha=0.8, zorder=3, label="Running best")

# Label kept experiments with description
for idx, loss in zip(kept_idx, kept_loss_vals):
    desc = str(valid.loc[idx, "description"]).strip()[:40]
    ax.annotate(desc, (idx, loss), textcoords="offset points",
                xytext=(6, -8), fontsize=7.5, color="#1a7a3a", alpha=0.85,
                rotation=25, ha="left", va="top")

n_total = len(df)
n_kept = len(df[df["status"] == "KEEP"])
ax.set_xlabel("Experiment #", fontsize=12)
ax.set_ylabel("Avg Guidance Loss (lower is better)", fontsize=12)
ax.set_title(f"PGDiff Autoresearch: {n_total} Experiments, {n_kept} Kept", fontsize=14)
ax.legend(loc="upper right", fontsize=8)
ax.grid(True, alpha=0.15)

best_loss = kept_loss_vals.min()
margin = max((baseline_loss - best_loss) * 0.15, abs(best_loss) * 0.05)
ax.set_ylim(best_loss - margin, baseline_loss + margin)
plt.tight_layout()
plt.savefig("progress.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved to progress.png")

## Per-Task Evolution

In [ ]:
tasks = df["task"].unique()
n_tasks = len(tasks)
fig, axes = plt.subplots(n_tasks, 1, figsize=(18, 5 * n_tasks), sharex=False)
if n_tasks == 1:
    axes = [axes]

for ax, task in zip(axes, tasks):
    task_df = df[(df["task"] == task) & (df["status"] != "CRASH")].copy().reset_index(drop=True)
    if len(task_df) == 0:
        continue
    disc = task_df[task_df["status"] == "DISCARD"]
    kept = task_df[task_df["status"] == "KEEP"]
    ax.scatter(disc.index, disc["avg_loss"], c="#cccccc", s=15, alpha=0.5, zorder=2)
    ax.scatter(kept.index, kept["avg_loss"], c="#2ecc71", s=60, zorder=4,
               edgecolors="black", linewidths=0.5)
    if len(kept) > 0:
        ax.step(kept.index, kept["avg_loss"].cummin(), where="post",
                color="#27ae60", linewidth=2, alpha=0.7, zorder=3)
    for idx, row in kept.iterrows():
        desc = str(row["description"])[:35]
        ax.annotate(desc, (idx, row["avg_loss"]), textcoords="offset points",
                    xytext=(5, -6), fontsize=7, color="#1a7a3a", alpha=0.8, rotation=20)
    ax.set_title(f"Task: {task}", fontsize=13)
    ax.set_ylabel("Avg Guidance Loss")
    ax.grid(True, alpha=0.15)

axes[-1].set_xlabel("Experiment # (within task)", fontsize=12)
plt.tight_layout()
plt.savefig("progress_by_task.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved to progress_by_task.png")

## Summary Statistics

In [ ]:
kept = df[df["status"] == "KEEP"].copy()
baseline_loss = df.iloc[0]["avg_loss"]
best_loss = kept["avg_loss"].min()
best_row = kept.loc[kept["avg_loss"].idxmin()]

print(f"Baseline avg_loss:     {baseline_loss:.1f}")
print(f"Best avg_loss:         {best_loss:.1f}")
improvement = baseline_loss - best_loss
print(f"Total improvement:     {improvement:.1f} ({improvement / baseline_loss * 100:.2f}%)")
print(f"Best experiment:       {best_row['description']}")
print(f"Best task:             {best_row['task']}")
print(f"Best params:           {best_row['key_params']}")
print()

for task in df["task"].unique():
    task_kept = kept[kept["task"] == task]
    if len(task_kept) > 0:
        print(f"Task {task:22s}: {len(task_kept):2d} kept, best={task_kept['avg_loss'].min():.1f}")

## Top Hits (Biggest Improvements)

In [ ]:
kept = df[df["status"] == "KEEP"].copy()
kept["prev_loss"] = kept["avg_loss"].shift(1)
kept["delta"] = kept["prev_loss"] - kept["avg_loss"]

hits = kept.iloc[1:].copy().sort_values("delta", ascending=False)

print(f"{'Rank':>4}  {'Delta':>10}  {'Loss':>10}  {'Task':<22}  {'Key Params':<30}  Description")
print("-" * 110)
for rank, (_, row) in enumerate(hits.iterrows(), 1):
    print(f"{rank:4d}  {row['delta']:+10.1f}  {row['avg_loss']:10.1f}  "
          f"{row['task']:22s}  {str(row['key_params'])[:30]:30s}  {row['description']}")

print(f"\n{'':>4}  {hits['delta'].sum():+10.1f}  {'':>10}  TOTAL improvement over baseline")

## Loss Breakdown by Experiment

Parse the `loss_breakdown` column to see per-term contribution.

In [ ]:
def parse_breakdown(raw):
    if pd.isna(raw) or not raw or raw == "N/A":
        return {}
    result = {}
    for pair in str(raw).split(","):
        if ":" in pair:
            k, v = pair.split(":", 1)
            try:
                result[k.strip()] = float(v.strip())
            except ValueError:
                pass
    return result

# Parse breakdowns for kept experiments
kept = df[df["status"] == "KEEP"].copy()
all_terms = set()
breakdowns = []
for _, row in kept.iterrows():
    bd = parse_breakdown(row.get("loss_breakdown", ""))
    breakdowns.append(bd)
    all_terms.update(bd.keys())

if all_terms:
    print(f"Loss terms found: {sorted(all_terms)}")
    print()
    for term in sorted(all_terms):
        values = [bd[term] for bd in breakdowns if term in bd]
        if values:
            print(f"  {term:25s}: min={min(values):.1f}  max={max(values):.1f}  mean={np.mean(values):.1f}  (n={len(values)})")
else:
    print("No loss breakdown data available (add loss_breakdown column to TSV)")